# Toy Agent Comparison

This notebook now uses the actual repo agents and the repo evaluation path on `TopKEnv`.
Read the outputs here as a mechanism study, not as the main paper benchmark pipeline.

The main checks in this version are:
- full-feedback ceilings versus the strongest partial-feedback baselines
- phased SPO versus phased SPO with margin-only exploration
- phased MSE controls, including the same margin-only exploration rule
- per-round regret, prediction accuracy, and decision-boundary diagnostics
- multi-arm scaling for the promising phased variants

In [1]:
from pathlib import Path
import sys

ROOT = Path.cwd()
if not (ROOT / 'src').exists() and (ROOT.parent / 'src').exists():
    ROOT = ROOT.parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

import numpy as np
import pandas as pd
from IPython.display import display
from lets_plot import LetsPlot

from notebooks.toy_agent_comparison_helpers import (
    default_dr_replay_configs,
    default_empucb_configs,
    default_main_configs,
    default_structured_repair_configs,
    default_structured_scale_configs,
    plot_diagnostics,
    plot_crossing_trace,
    plot_instant_regret,
    plot_prediction_accuracy,
    plot_prediction_profiles,
    plot_regret,
    print_regret_summary,
    run_crossing_trace_suite,
    run_scale_cases,
    run_suite,
)

LetsPlot.setup_html()
SMOOTH_WINDOW = 31


In [2]:
W_2ARM = np.array([[1.0], [1.5]], dtype=float)
main_configs = default_main_configs(include_mse=True, include_pfy=False)
structured_configs = default_structured_repair_configs()
empucb_configs = default_empucb_configs()
dr_configs = default_dr_replay_configs()
display(pd.DataFrame([{'group': 'main', 'label': label, **cfg} for label, cfg in main_configs]))
display(pd.DataFrame([{'group': 'structured', 'label': label, **cfg} for label, cfg in structured_configs]))
display(pd.DataFrame([{'group': 'empucb', 'label': label, **cfg} for label, cfg in empucb_configs]))
display(pd.DataFrame([{'group': 'dr_replay', 'label': label, **cfg} for label, cfg in dr_configs]))


,group,label,agent_name,full_feedback,mask_grad,phased,margin_explore
0,main,BayesSPO (full),BayesianSPO,True,True,NaN,NaN
1,main,"BayesSPO (partial, masked)",BayesianSPO,False,True,NaN,NaN
2,main,BayesSPO-Phased (partial),BayesianSPO,False,True,True,NaN
3,main,BayesSPO-Phased (margin only),StructuredRepairSPO,False,True,True,True
4,main,BayesianMSE (full),BayesianMSE,True,False,NaN,NaN
5,main,BayesianMSE-Phased (partial),BayesianMSE,False,False,True,NaN
6,main,BayesianMSE-Phased (margin only),StructuredRepairMSE,False,False,True,True


,group,label,agent_name,full_feedback,mask_grad,phased,margin_explore,phase_length,phase_epochs
0,structured,BayesSPO (full),BayesianSPO,True,True,NaN,NaN,NaN,NaN
1,structured,"BayesSPO (partial, masked)",BayesianSPO,False,True,NaN,NaN,NaN,NaN
2,structured,BayesSPO-Phased (partial),BayesianSPO,False,True,True,NaN,NaN,NaN
3,structured,"BayesSPO (partial, margin explore)",StructuredRepairSPO,False,True,NaN,True,50.0,5.0
4,structured,BayesSPO-Phased (margin only),StructuredRepairSPO,False,True,True,True,NaN,NaN
5,structured,BayesianMSE-Phased (margin only),StructuredRepairMSE,False,False,True,True,NaN,NaN


,group,label,agent_name,full_feedback,mask_grad,phased
0,empucb,BayesSPO (full),BayesianSPO,True,True,NaN
1,empucb,"BayesSPO (partial, masked)",BayesianSPO,False,True,NaN
2,empucb,BayesSPO-Phased (partial),BayesianSPO,False,True,True
3,empucb,SPO-EmpUCB (full),SPOEmpUCB,True,True,NaN
4,empucb,SPO-EmpUCB (partial),SPOEmpUCB,False,True,NaN
5,empucb,SPO-EmpUCB-Phased (partial),SPOEmpUCB,False,True,True


,group,label,agent_name,full_feedback,mask_grad,phased,dr_lambda,dr_clip,dr_samples
0,dr_replay,BayesSPO (full),BayesianSPO,True,True,NaN,NaN,NaN,NaN
1,dr_replay,"BayesSPO (partial, masked)",BayesianSPO,False,True,NaN,NaN,NaN,NaN
2,dr_replay,BayesSPO-Phased (partial),BayesianSPO,False,True,True,NaN,NaN,NaN
3,dr_replay,BayesSPO-Phased (DR),DRReplaySPO,False,True,True,0.1,10.0,64.0
4,dr_replay,"BayesSPO-Phased (DR, clip=2)",DRReplaySPO,False,True,True,0.1,2.0,64.0


## 2-arm linear

This is the easy, well-specified case. If the bottleneck story is right, full feedback should dominate, but partial-feedback SPO should not necessarily collapse.

In [3]:
linear_rounds, linear_summary, linear_artifacts = run_suite(
    W_true=W_2ARM,
    k=1,
    configs=main_configs,
    n_rounds=500,
    n_seeds=5,
    hidden_dim=8,
    cost_fn='linear',
    capture_agents=True,
)
display(linear_summary)
print_regret_summary(linear_summary)
display(plot_regret(linear_rounds, '2-arm linear regret', smooth_window=SMOOTH_WINDOW))
display(plot_instant_regret(linear_rounds, '2-arm linear instant regret', smooth_window=SMOOTH_WINDOW))
display(plot_prediction_accuracy(linear_rounds, '2-arm linear prediction accuracy', smooth_window=SMOOTH_WINDOW))
display(plot_diagnostics(linear_rounds, '2-arm linear diagnostics', smooth_window=SMOOTH_WINDOW))
display(plot_prediction_profiles(linear_artifacts, W_2ARM, cost_fn='linear'))


,agent,regret_mean,regret_std,decision_rate,mean_overall_mae,mean_flip_size,mean_contested_mae,mean_contested_sigma,mean_rank_corr
0,BayesSPO (full),0.028519,0.060973,0.9968,0.082943,0.0616,0.082149,0.062761,0.995190
1,BayesianMSE (full),0.028856,0.053668,0.9960,0.061661,0.0640,0.061613,0.064012,0.993587
2,BayesSPO-Phased (partial),1.729937,1.068811,0.9912,0.132794,0.0672,0.132831,0.083268,0.983968
3,BayesianMSE-Phased (partial),1.748081,1.068922,0.9868,0.115854,0.1016,0.112207,0.084699,0.975150
4,BayesSPO-Phased (margin only),2.165366,1.135282,0.9868,0.132849,0.0672,0.132867,0.083268,0.975150
5,"BayesSPO (partial, masked)",2.175411,1.142655,0.9832,0.126505,0.0824,0.124346,0.084687,0.967936
6,BayesianMSE-Phased (margin only),2.183510,1.131589,0.9824,0.115754,0.0992,0.112200,0.084616,0.966333


BayesSPO (full)               : regret = 0.0 +/- 0.1
BayesianMSE (full)            : regret = 0.0 +/- 0.1
BayesSPO-Phased (partial)     : regret = 1.7 +/- 1.1
BayesianMSE-Phased (partial)  : regret = 1.7 +/- 1.1
BayesSPO-Phased (margin only) : regret = 2.2 +/- 1.1
BayesSPO (partial, masked)    : regret = 2.2 +/- 1.1
BayesianMSE-Phased (margin only): regret = 2.2 +/- 1.1


## 2-arm sigmoid

This is the misspecified case. The main questions are whether phasing rescues partial SPO by itself, whether margin-only exploration is the real active ingredient, and whether the same exploration rule also helps phased MSE or if the gain is specifically decision-focused.

In [4]:
sigmoid_rounds, sigmoid_summary, sigmoid_artifacts = run_suite(
    W_true=W_2ARM,
    k=1,
    configs=main_configs,
    n_rounds=500,
    n_seeds=5,
    hidden_dim=8,
    cost_fn='sigmoid',
    capture_agents=True,
)
display(sigmoid_summary)
print_regret_summary(sigmoid_summary)
display(plot_regret(sigmoid_rounds, '2-arm sigmoid regret', smooth_window=SMOOTH_WINDOW))
display(plot_instant_regret(sigmoid_rounds, '2-arm sigmoid instant regret', smooth_window=SMOOTH_WINDOW))
display(plot_prediction_accuracy(sigmoid_rounds, '2-arm sigmoid prediction accuracy', smooth_window=SMOOTH_WINDOW))
display(plot_diagnostics(sigmoid_rounds, '2-arm sigmoid diagnostics', smooth_window=SMOOTH_WINDOW))
display(plot_prediction_profiles(sigmoid_artifacts, W_2ARM, cost_fn='sigmoid'))


,agent,regret_mean,regret_std,decision_rate,mean_overall_mae,mean_flip_size,mean_contested_mae,mean_contested_sigma,mean_rank_corr
0,BayesSPO (full),0.007834,0.014645,0.9956,0.501074,0.6408,0.493524,0.064343,0.992786
1,BayesianMSE (full),0.214252,0.205122,0.9508,0.276346,0.6976,0.267097,0.084857,0.903006
2,BayesSPO-Phased (margin only),4.615534,0.634809,0.7916,0.155603,1.8264,0.134500,0.108439,0.583968
3,BayesianMSE-Phased (margin only),5.318093,0.654942,0.7712,0.165116,1.8680,0.142428,0.108434,0.543086
4,BayesSPO-Phased (partial),10.113685,1.380480,0.8056,0.150654,1.4384,0.135753,0.109506,0.612024
5,BayesianMSE-Phased (partial),10.174635,1.154786,0.8136,0.160710,1.5192,0.143701,0.109259,0.628056
6,"BayesSPO (partial, masked)",14.056453,1.309703,0.6920,0.320956,1.9592,0.293107,0.106161,0.384369


BayesSPO (full)               : regret = 0.0 +/- 0.0
BayesianMSE (full)            : regret = 0.2 +/- 0.2
BayesSPO-Phased (margin only) : regret = 4.6 +/- 0.6
BayesianMSE-Phased (margin only): regret = 5.3 +/- 0.7
BayesSPO-Phased (partial)     : regret = 10.1 +/- 1.4
BayesianMSE-Phased (partial)  : regret = 10.2 +/- 1.2
BayesSPO (partial, masked)    : regret = 14.1 +/- 1.3


### Crossing Trace And Frozen Test Regret

This focuses on the SPO family only. The crossing trace tracks the estimated decision boundary over training, and the frozen test summary evaluates the final predictor greedily without UCB exploration.

In [5]:
spo_trace_configs = [item for item in main_configs if 'BayesSPO' in item[0]]
sigmoid_trace, sigmoid_frozen_summary, sigmoid_true_crossing = run_crossing_trace_suite(
    W_true=W_2ARM,
    k=1,
    configs=spo_trace_configs,
    n_rounds=500,
    hidden_dim=8,
    cost_fn='sigmoid',
    checkpoint_every=10,
)
display(sigmoid_frozen_summary)
display(plot_crossing_trace(sigmoid_trace, sigmoid_true_crossing, '2-arm sigmoid crossing trace'))


,agent,final_crossing,crossing_error,frozen_test_regret_mean,frozen_test_regret_std,frozen_decision_rate
0,BayesSPO (full),0.003047,0.003047,0.000000,0.000000,1.0000
1,BayesSPO-Phased (partial),-0.423427,0.423427,0.004332,0.011222,0.8310
2,BayesSPO-Phased (margin only),0.507450,0.507450,0.005743,0.013759,0.8060
3,"BayesSPO (partial, masked)",-1.068735,1.068735,0.017758,0.028620,0.6445


### Margin-Only Ablation

This keeps the pieces that still look worth testing after trimming the dead ends: plain phased partial SPO, online SPO margin exploration, phased SPO margin-only replay, and the phased MSE controls with and without the same margin rule.

In [6]:
structured_rounds, structured_summary, structured_artifacts = run_suite(
    W_true=W_2ARM,
    k=1,
    configs=structured_configs,
    n_rounds=500,
    n_seeds=5,
    hidden_dim=8,
    cost_fn='sigmoid',
    capture_agents=True,
)
display(structured_summary)
print_regret_summary(structured_summary)
display(plot_regret(structured_rounds, '2-arm sigmoid structured repair regret', smooth_window=SMOOTH_WINDOW))
display(plot_instant_regret(structured_rounds, '2-arm sigmoid masking and margin instant regret', smooth_window=SMOOTH_WINDOW))
display(plot_prediction_accuracy(structured_rounds, '2-arm sigmoid masking and margin prediction accuracy', smooth_window=SMOOTH_WINDOW))
display(plot_diagnostics(structured_rounds, '2-arm sigmoid structured repair diagnostics', smooth_window=SMOOTH_WINDOW))
display(plot_prediction_profiles(structured_artifacts, W_2ARM, cost_fn='sigmoid'))


,agent,regret_mean,regret_std,decision_rate,mean_overall_mae,mean_flip_size,mean_contested_mae,mean_contested_sigma,mean_rank_corr
0,BayesSPO (full),0.007834,0.014645,0.9956,0.501074,0.6408,0.493524,0.064343,0.992786
1,BayesSPO-Phased (margin only),4.615534,0.634809,0.7916,0.155603,1.8264,0.134500,0.108439,0.583968
2,BayesianMSE-Phased (margin only),5.318093,0.654942,0.7712,0.165116,1.8680,0.142428,0.108434,0.543086
3,BayesSPO-Phased (partial),10.113685,1.380480,0.8056,0.150654,1.4384,0.135753,0.109506,0.612024
4,"BayesSPO (partial, margin explore)",12.252804,1.017348,0.5716,0.321673,1.9712,0.292700,0.106197,0.143086
5,"BayesSPO (partial, masked)",14.056453,1.309703,0.6920,0.320956,1.9592,0.293107,0.106161,0.384369


BayesSPO (full)               : regret = 0.0 +/- 0.0
BayesSPO-Phased (margin only) : regret = 4.6 +/- 0.6
BayesianMSE-Phased (margin only): regret = 5.3 +/- 0.7
BayesSPO-Phased (partial)     : regret = 10.1 +/- 1.4
BayesSPO (partial, margin explore): regret = 12.3 +/- 1.0
BayesSPO (partial, masked)    : regret = 14.1 +/- 1.3


In [7]:
structured_trace, structured_frozen_summary, structured_true_crossing = run_crossing_trace_suite(
    W_true=W_2ARM,
    k=1,
    configs=structured_configs,
    n_rounds=500,
    hidden_dim=8,
    cost_fn='sigmoid',
    checkpoint_every=10,
)
display(structured_frozen_summary)
display(plot_crossing_trace(structured_trace, structured_true_crossing, '2-arm sigmoid structured repair crossing trace'))


,agent,final_crossing,crossing_error,frozen_test_regret_mean,frozen_test_regret_std,frozen_decision_rate
0,BayesSPO (full),0.003047,0.003047,0.000000,0.000000,1.0000
1,"BayesSPO (partial, margin explore)",0.372481,0.372481,0.003200,0.009130,0.8565
2,BayesSPO-Phased (partial),-0.423427,0.423427,0.004332,0.011222,0.8310
3,BayesSPO-Phased (margin only),0.507450,0.507450,0.005743,0.013759,0.8060
4,BayesianMSE-Phased (margin only),0.709519,0.709519,0.009289,0.019180,0.7515
5,"BayesSPO (partial, masked)",-1.068735,1.068735,0.017758,0.028620,0.6445


### EmpUCB Last-Layer Ablation

This isolates whether the Bayesian last layer matters. `SPO-EmpUCB` keeps the SPO update but replaces the Bayesian posterior head with a standard learned linear head plus empirical count-based UCB bonus.


In [8]:
empucb_rounds, empucb_summary, empucb_artifacts = run_suite(
    W_true=W_2ARM,
    k=1,
    configs=empucb_configs,
    n_rounds=500,
    n_seeds=5,
    hidden_dim=8,
    cost_fn='sigmoid',
    capture_agents=True,
)
display(empucb_summary)
print_regret_summary(empucb_summary)
display(plot_regret(empucb_rounds, '2-arm sigmoid: Bayesian last layer vs EmpUCB', smooth_window=SMOOTH_WINDOW))
display(plot_instant_regret(empucb_rounds, '2-arm sigmoid instant regret: Bayesian last layer vs EmpUCB', smooth_window=SMOOTH_WINDOW))
display(plot_prediction_accuracy(empucb_rounds, '2-arm sigmoid prediction accuracy: Bayesian last layer vs EmpUCB', smooth_window=SMOOTH_WINDOW))
display(plot_diagnostics(empucb_rounds, '2-arm sigmoid diagnostics: Bayesian last layer vs EmpUCB', smooth_window=SMOOTH_WINDOW))
display(plot_prediction_profiles(
    empucb_artifacts,
    W_true=W_2ARM,
    cost_fn='sigmoid',
    title='2-arm sigmoid final predictions: Bayesian last layer vs EmpUCB',
))


,agent,regret_mean,regret_std,decision_rate,mean_overall_mae,mean_flip_size,mean_contested_mae,mean_contested_sigma,mean_rank_corr
0,BayesSPO (full),0.007834,0.014645,0.9956,0.501074,0.6408,0.493524,0.064343,0.992786
1,SPO-EmpUCB (full),0.181617,0.278726,0.9860,0.497493,1.9992,0.468763,0.181805,0.973542
2,BayesSPO-Phased (partial),10.113685,1.380480,0.8056,0.150654,1.4384,0.135753,0.109506,0.612024
3,SPO-EmpUCB (partial),13.830710,1.101368,0.5808,0.494572,1.9992,0.465099,0.255475,0.161523
4,"BayesSPO (partial, masked)",14.056453,1.309703,0.6920,0.320956,1.9592,0.293107,0.106161,0.384369
5,SPO-EmpUCB-Phased (partial),15.507082,0.484613,0.5188,0.410908,1.9776,0.381742,0.255475,0.037275


BayesSPO (full)               : regret = 0.0 +/- 0.0
SPO-EmpUCB (full)             : regret = 0.2 +/- 0.3
BayesSPO-Phased (partial)     : regret = 10.1 +/- 1.4
SPO-EmpUCB (partial)          : regret = 13.8 +/- 1.1
BayesSPO (partial, masked)    : regret = 14.1 +/- 1.3
SPO-EmpUCB-Phased (partial)   : regret = 15.5 +/- 0.5


### DR Replay Ablation

This keeps `BayesianSPO` fixed and changes only phased replay: observed residuals are upweighted by estimated inclusion propensity with `lambda` flooring and clipping, in the spirit of a Hu-Kallus-style DR correction.


In [9]:
dr_rounds, dr_summary, dr_artifacts = run_suite(
    W_true=W_2ARM,
    k=1,
    configs=dr_configs,
    n_rounds=500,
    n_seeds=5,
    hidden_dim=8,
    cost_fn='sigmoid',
    capture_agents=True,
)
display(dr_summary)
print_regret_summary(dr_summary)
display(plot_regret(dr_rounds, '2-arm sigmoid: DR replay ablation', smooth_window=SMOOTH_WINDOW))
display(plot_instant_regret(dr_rounds, '2-arm sigmoid instant regret: DR replay ablation', smooth_window=SMOOTH_WINDOW))
display(plot_prediction_accuracy(dr_rounds, '2-arm sigmoid prediction accuracy: DR replay ablation', smooth_window=SMOOTH_WINDOW))
display(plot_diagnostics(dr_rounds, '2-arm sigmoid diagnostics: DR replay ablation', smooth_window=SMOOTH_WINDOW))
display(plot_prediction_profiles(
    dr_artifacts,
    W_true=W_2ARM,
    cost_fn='sigmoid',
    title='2-arm sigmoid final predictions: DR replay ablation',
))


,agent,regret_mean,regret_std,decision_rate,mean_overall_mae,mean_flip_size,mean_contested_mae,mean_contested_sigma,mean_rank_corr
0,BayesSPO (full),0.007834,0.014645,0.9956,0.501074,0.6408,0.493524,0.064343,0.992786
1,"BayesSPO-Phased (DR, clip=2)",10.084387,1.415542,0.8052,0.150494,1.4288,0.135672,0.109512,0.611222
2,BayesSPO-Phased (DR),10.097691,1.415430,0.8048,0.150466,1.4280,0.135676,0.109518,0.610421
3,BayesSPO-Phased (partial),10.113685,1.380480,0.8056,0.150654,1.4384,0.135753,0.109506,0.612024
4,"BayesSPO (partial, masked)",14.056453,1.309703,0.6920,0.320956,1.9592,0.293107,0.106161,0.384369


BayesSPO (full)               : regret = 0.0 +/- 0.0
BayesSPO-Phased (DR, clip=2)  : regret = 10.1 +/- 1.4
BayesSPO-Phased (DR)          : regret = 10.1 +/- 1.4
BayesSPO-Phased (partial)     : regret = 10.1 +/- 1.4
BayesSPO (partial, masked)    : regret = 14.1 +/- 1.3


## Multi-arm Structured Scaling

These cases keep the smaller active shortlist and test whether phased SPO margin-only still helps once the toy moves beyond 2 arms, whether the same benefit survives a DR-style phased replay correction, and whether that effect transfers to phased MSE.


In [10]:
scale_configs = default_structured_scale_configs()
display(pd.DataFrame([{'group': 'scale', 'label': label, **cfg} for label, cfg in scale_configs]))
scale_cases = [
    dict(label='4 arms, k=2', n_arms=4, k=2, context_dim=5, hidden_dim=8, n_rounds=500, n_seeds=3, weight_seed=42),
    dict(label='8 arms, k=4', n_arms=8, k=4, context_dim=5, hidden_dim=8, n_rounds=500, n_seeds=3, weight_seed=43),
    dict(label='16 arms, k=4', n_arms=16, k=4, context_dim=5, hidden_dim=16, n_rounds=500, n_seeds=3, weight_seed=44),
]
scale_results = run_scale_cases(scale_cases, scale_configs, cost_fn='sigmoid')
for label, case in scale_results.items():
    print(label)
    display(case['summary'])
    print_regret_summary(case['summary'])
    print()


,group,label,agent_name,full_feedback,mask_grad,phased,dr_lambda,dr_clip,dr_samples,margin_explore
0,scale,BayesSPO (full),BayesianSPO,True,True,NaN,NaN,NaN,NaN,NaN
1,scale,BayesSPO-Phased (partial),BayesianSPO,False,True,True,NaN,NaN,NaN,NaN
2,scale,BayesSPO-Phased (DR),DRReplaySPO,False,True,True,0.1,10.0,64.0,NaN
3,scale,BayesSPO-Phased (margin only),StructuredRepairSPO,False,True,True,NaN,NaN,NaN,True
4,scale,BayesianMSE-Phased (partial),BayesianMSE,False,False,True,NaN,NaN,NaN,NaN
5,scale,BayesianMSE-Phased (margin only),StructuredRepairMSE,False,False,True,NaN,NaN,NaN,True


4 arms, k=2


,agent,regret_mean,regret_std,decision_rate,mean_overall_mae,mean_flip_size,mean_contested_mae,mean_contested_sigma,mean_rank_corr
0,BayesSPO (full),1.797498,0.860302,0.840667,0.492144,1.068000,0.487786,0.160804,0.886840
1,BayesSPO-Phased (margin only),13.365271,1.955481,0.584000,0.208661,2.673333,0.192124,0.219785,0.601042
2,BayesianMSE-Phased (margin only),13.861938,2.141028,0.572000,0.212035,2.648000,0.195402,0.219708,0.614268
3,BayesSPO-Phased (DR),16.193095,2.348929,0.575333,0.207247,2.509333,0.191995,0.215102,0.604515
4,BayesianMSE-Phased (partial),16.199360,1.662562,0.574000,0.210604,2.489333,0.195112,0.214842,0.616940
5,BayesSPO-Phased (partial),16.329840,2.765846,0.563333,0.206497,2.504000,0.191483,0.215374,0.600107


BayesSPO (full)               : regret = 1.8 +/- 0.9
BayesSPO-Phased (margin only) : regret = 13.4 +/- 2.0
BayesianMSE-Phased (margin only): regret = 13.9 +/- 2.1
BayesSPO-Phased (DR)          : regret = 16.2 +/- 2.3
BayesianMSE-Phased (partial)  : regret = 16.2 +/- 1.7
BayesSPO-Phased (partial)     : regret = 16.3 +/- 2.8

8 arms, k=4


,agent,regret_mean,regret_std,decision_rate,mean_overall_mae,mean_flip_size,mean_contested_mae,mean_contested_sigma,mean_rank_corr
0,BayesSPO (full),6.593660,0.671865,0.554000,0.489992,2.777333,0.489827,0.161445,0.860992
1,BayesSPO-Phased (margin only),31.880566,3.585533,0.308667,0.211100,5.088000,0.206172,0.220569,0.660655
2,BayesianMSE-Phased (margin only),33.500179,2.363280,0.284000,0.217250,5.196000,0.212021,0.220271,0.652925
3,BayesSPO-Phased (partial),35.280924,1.520341,0.305333,0.209949,4.948000,0.205694,0.215870,0.668003
4,BayesianMSE-Phased (partial),35.587685,0.759013,0.308667,0.216737,5.058667,0.212337,0.215845,0.665219
5,BayesSPO-Phased (DR),35.687927,0.408899,0.300000,0.211467,5.041333,0.206541,0.215592,0.663279


BayesSPO (full)               : regret = 6.6 +/- 0.7
BayesSPO-Phased (margin only) : regret = 31.9 +/- 3.6
BayesianMSE-Phased (margin only): regret = 33.5 +/- 2.4
BayesSPO-Phased (partial)     : regret = 35.3 +/- 1.5
BayesianMSE-Phased (partial)  : regret = 35.6 +/- 0.8
BayesSPO-Phased (DR)          : regret = 35.7 +/- 0.4

16 arms, k=4


,agent,regret_mean,regret_std,decision_rate,mean_overall_mae,mean_flip_size,mean_contested_mae,mean_contested_sigma,mean_rank_corr
0,BayesSPO (full),6.854922,0.898933,0.490667,0.495903,2.670667,0.495834,0.176085,0.931932
1,BayesSPO-Phased (margin only),62.635404,4.508583,0.164667,0.201396,6.622667,0.196065,0.357631,0.654405
2,BayesianMSE-Phased (margin only),64.686730,6.025622,0.165333,0.209109,6.677333,0.204307,0.353954,0.643886
3,BayesSPO-Phased (partial),77.357458,1.349647,0.172667,0.198948,6.124000,0.193951,0.340522,0.691561
4,BayesSPO-Phased (DR),77.773586,3.072196,0.169333,0.199837,6.114667,0.194627,0.340572,0.703799
5,BayesianMSE-Phased (partial),79.661469,1.779973,0.175333,0.205781,6.144000,0.200739,0.339315,0.688877


BayesSPO (full)               : regret = 6.9 +/- 0.9
BayesSPO-Phased (margin only) : regret = 62.6 +/- 4.5
BayesianMSE-Phased (margin only): regret = 64.7 +/- 6.0
BayesSPO-Phased (partial)     : regret = 77.4 +/- 1.3
BayesSPO-Phased (DR)          : regret = 77.8 +/- 3.1
BayesianMSE-Phased (partial)  : regret = 79.7 +/- 1.8



In [11]:
focus_cases = ['8 arms, k=4', '16 arms, k=4']
for focus_case in focus_cases:
    display(plot_regret(scale_results[focus_case]['rounds'], f'{focus_case} regret', smooth_window=SMOOTH_WINDOW))
    display(plot_diagnostics(scale_results[focus_case]['rounds'], f'{focus_case} diagnostics', smooth_window=SMOOTH_WINDOW))
